# Kiểm tra luồng đọc cache — CPU ONLY

**Accelerator: None**, Internet ON, Save & Run All. Không GPU, không API token, không huấn luyện hay sinh caption.

Gắn hai Dataset giống full baseline: `ducanh2403/visual-cache` và `vuthetam/mscoco-2014`. Giữ cùng2.048captionevents TRAIN của64globalbatches trước;4tiến trình đọc/CPU LRU256. Hai lượt đọc mountedH5 → chép riêng mẫu theo thứ tự vật lý sang localfloat16NPY → hai lượt localmemmap. Kiểm tra hash toàn luồng đặc trưng khớp; không mất thông tin.

Chỉ chép mẫu khoảng0,6GB, không phải full35GB. Ghi disk/RAM/staging/startup/read+stack/walltime; CPUdelivery không phải thời gian GPUupdate. Không xóa OS cache: không gọi là phép đo cold-cache, cachelocal nhỏ có thể nằmRAM; chưa suy ra tốc độfull hay ETA10epoch.

Giới hạn30phút là capdừng; không cam kết runtime. Gửi `clip_patch_cache_io_probe_v1_review.zip`; chưa chạy lại fullGPU. Tác giả chỉ kiểm NumPy/mock/AST; phép đo Kaggle thực do bạn chạy.


In [ ]:
import os,sys,json,hashlib,shutil,time,tempfile,urllib.request,zipfile,traceback,queue,signal,subprocess,threading
from pathlib import Path
REVISION='a3a927ba3c27cbad9bbe77b76fc0fe7cb41e0f40'
MAPPING={'experiments/clip_patch_baseline/io_probe.py': 'io_probe.py', 'experiments/clip_patch_baseline/test_io_probe.py': 'test_io_probe.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/IO_PROBE_PROTOCOL.md': 'IO_PROBE_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py'}
HASHES={'experiments/clip_patch_baseline/io_probe.py': '66ebf985ac6885372a23ce6c4d36ed2d88f2581a00ac43e2321e065c8e240d17', 'experiments/clip_patch_baseline/test_io_probe.py': '676d1e441cc7704df9c06c870d6c12a3201e952fc0a9236dcd4bc436b89eb746', 'experiments/clip_patch_baseline/full_data.py': '2286ace2dad228b3e7fff8e2c411ab212f1d931ce59228d8ad994d45f5581115', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/IO_PROBE_PROTOCOL.md': '57996fbe5b960162f9a8eb92b37b0ebf9d4406f1a461f06f09b02fc9464fea3b', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/clip_patch_cache_io_probe_v1')
for p in (VISUAL_CACHE,COCO_JSON):
    if not p.exists():raise FileNotFoundError('Attach/correct input path: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve old output; use fresh session or output path')
if shutil.disk_usage('/tmp').free<4*1024**3:raise RuntimeError('Need4GiB scratch free for metadata/sample; do not copy fullcache')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='patch-cache-io-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+1800
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):os.environ[key]='1'
os.environ['CUDA_VISIBLE_DEVICES']='';os.environ['PYTHONPATH']=str(SOURCE)
CONFIG=SOURCE/'config.json'
cfg=dict(output=str(OUTPUT),stage=str(SOURCE/'metadata'),local=str(SOURCE/'local_sample'),visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),wall_seconds=1740)
CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('CPU parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CPU cache IO active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('CPU IO parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for path,name in MAPPING.items():
        raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,timeout=90).read()
        if hashlib.sha256(raw).hexdigest()!=HASHES[path]:raise ValueError('Changed source: '+name)
        (SOURCE/name).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_io_probe.py'],120)
    cfg['wall_seconds']=PARENT_DEADLINE-time.monotonic()-30;CONFIG.write_text(json.dumps(cfg,indent=2))
    run([sys.executable,'-u',SOURCE/'io_probe.py','--config',CONFIG],cfg['wall_seconds'])
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CPU_PATCH_IO_PROBE_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,models_executed=False,test_features_read=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True)
    print('CPU IO COMPLETE; do not automatically start fullGPU.' if complete else 'INCOMPLETE; preserve log and send review, no automatic retry.',flush=True)
